<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [ ]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [4]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [6]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [20]:
nasa_coordinate = [
    29.559684888503615,
    -95.0830971930759
]

site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=5
)

for _, site in launch_sites_df.iterrows():
    coordinate = [
        float(site["Lat"]),
        float(site["Long"])
    ]

    site_name = site["Launch Site"]

    folium.Circle(
        location=coordinate,
        radius=1000,
        color="#d35400",
        fill=True,
        fill_opacity=0.3,
        popup=site_name
    ).add_to(site_map)

    folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(180, 30),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:12px; '
                'color:#d35400; white-space:nowrap;">'
                '<b>{}</b></div>'
            ).format(site_name)
        )
    ).add_to(site_map)

site_map.fit_bounds(
    launch_sites_df[["Lat", "Long"]].values.tolist()
)

site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [8]:
spacex_df["marker_color"] = spacex_df["class"].map({
    1: "green",
    0: "red"
})

spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [9]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [10]:
marker_cluster = MarkerCluster(
    name="Launch Outcomes"
).add_to(site_map)

for index, record in spacex_df.iterrows():
    coordinate = [
        float(record["Lat"]),
        float(record["Long"])
    ]

    outcome = (
        "Success" if record["class"] == 1
        else "Failure"
    )

    folium.Marker(
        location=coordinate,
        popup=(
            "{}<br>Outcome: {}<br>Record: {}"
        ).format(
            record["Launch Site"],
            outcome,
            index
        ),
        icon=folium.Icon(
            color="white",
            icon_color=record["marker_color"],
            icon="info-sign"
        )
    ).add_to(marker_cluster)

site_map

*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [ ]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [11]:
formatter = (
    "function(num) {"
    "return L.Util.formatNum(num, 5);"
    "};"
)

MousePosition(
    position="topright",
    separator=" Long: ",
    empty_string="NaN",
    lng_first=False,
    prefix="Lat:",
    lat_formatter=formatter,
    lng_formatter=formatter
).add_to(site_map)

site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [12]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    a = min(1.0, max(0.0, a))

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [14]:
from math import sin, cos, sqrt, atan2, radians

coastline_coordinate = [
    28.56367,
    -80.57163
]

distances_to_coastline_point = launch_sites_df.apply(
    lambda row: calculate_distance(
        float(row["Lat"]),
        float(row["Long"]),
        coastline_coordinate[0],
        coastline_coordinate[1]
    ),
    axis=1
)

selected_site = launch_sites_df.loc[
    distances_to_coastline_point.idxmin()
]

launch_coordinate = [
    float(selected_site["Lat"]),
    float(selected_site["Long"])
]

print("Selected site:", selected_site["Launch Site"])
print("Launch coordinates:", launch_coordinate)

Selected site: CCAFS SLC-40
Launch coordinates: [28.56319718, -80.57682003]


In [15]:
def add_distance_to_point(label, latitude, longitude, color="purple"):
    point = [
        float(latitude),
        float(longitude)
    ]

    distance = calculate_distance(
        launch_coordinate[0],
        launch_coordinate[1],
        point[0],
        point[1]
    )

    folium.Marker(
        location=point,
        popup="{}: {:.2f} km".format(label, distance),
        icon=folium.Icon(
            color=color,
            icon="info-sign"
        )
    ).add_to(site_map)

    folium.Marker(
        location=point,
        icon=DivIcon(
            icon_size=(260, 30),
            icon_anchor=(0, -20),
            html=(
                '<div style="font-size:12px; '
                'color:{}; white-space:nowrap;">'
                '<b>{}: {:.2f} km</b></div>'
            ).format(color, label, distance)
        )
    ).add_to(site_map)

    folium.PolyLine(
        locations=[
            launch_coordinate,
            point
        ],
        color=color,
        weight=3,
        tooltip="{}: {:.2f} km".format(label, distance)
    ).add_to(site_map)

    return distance

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [16]:
distance_coastline = add_distance_to_point(
    "Selected coastline point",
    coastline_coordinate[0],
    coastline_coordinate[1],
    color="blue"
)

print(
    "Coastline distance: {:.2f} km".format(
        distance_coastline
    )
)

site_map.fit_bounds([
    launch_coordinate,
    coastline_coordinate
])

site_map

Coastline distance: 0.51 km


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site


In [17]:
nearby_points = {
    "Selected railway point": [None, None],
    "Selected highway point": [None, None],
    "Selected city point": [None, None]
}

distance_results = []

for label, coordinate in nearby_points.items():
    latitude, longitude = coordinate

    if latitude is None or longitude is None:
        print("Coordinates not entered:", label)
        continue

    distance = add_distance_to_point(
        label,
        latitude,
        longitude
    )

    distance_results.append({
        "Feature": label,
        "Latitude": latitude,
        "Longitude": longitude,
        "Distance_KM": distance
    })

if distance_results:
    display(pd.DataFrame(distance_results))

site_map

Coordinates not entered: Selected railway point
Coordinates not entered: Selected highway point
Coordinates not entered: Selected city point


In [18]:
site_summary = (
    spacex_df.groupby("Launch Site")["class"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "Launches",
        "sum": "Successful_Launches",
        "mean": "Success_Rate"
    })
)

site_summary["Success_Rate_Percent"] = (
    site_summary["Success_Rate"] * 100
)

site_summary

,Launches,Successful_Launches,Success_Rate,Success_Rate_Percent
Launch Site,,,,
CCAFS LC-40,26,7,0.269231,26.923077
CCAFS SLC-40,7,3,0.428571,42.857143
KSC LC-39A,13,10,0.769231,76.923077
VAFB SLC-4E,10,4,0.400000,40.000000


In [19]:
site_map.save("spacex_launch_sites_map.html")

print("Saved spacex_launch_sites_map.html")

Saved spacex_launch_sites_map.html


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
